In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import ast
import random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm
from tokenizers import ByteLevelBPETokenizer
from collections import Counter

# --- KONFIGURÁCIÓ v4 ---
D_MODEL = 512
MAX_LEN = 400
BATCH_SIZE = 32
CHECKPOINT_DIR = ""
DATA_PATH_CSV = ""
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. ÚJ ADATSTRUKTÚRA ---
def load_v4_data(path, max_recipes=150000):
    df = pd.read_csv(path).dropna(subset=['name', 'ingredients', 'steps', 'minutes']).head(max_recipes)
    recipes = []

    print("Strukturált adatok előkészítése (idővel)...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            ings_list = ast.literal_eval(row['ingredients'])
            steps_list = ast.literal_eval(row['steps'])
            # Az időt percekben adjuk meg
            cooking_time = int(row['minutes'])

            src = "ingredients: " + ", ".join(ings_list).lower()

            # ÚJ TARGET STRUKTÚRA: time: kulcsszóval
            tgt = (f"title: {row['name'].lower()} "
                   f"time: {cooking_time} mins "
                   f"ingredients: {', '.join(ings_list).lower()} "
                   f"steps: {' '.join(steps_list).lower()}")

            recipes.append({"src": src, "tgt": tgt, "ings_cnt": len(ings_list)})
        except: continue

    weights = [r['ings_cnt'] for r in recipes]
    return recipes, weights


In [ ]:
import os
import time
import torch
import random
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tokenizers import ByteLevelBPETokenizer
import torch.nn as nn # Added this import

# --- KONFIGURÁCIÓ v4 ---
D_MODEL = 512
MAX_LEN = 400
BATCH_SIZE = 32
CHECKPOINT_DIR = ""
DATA_PATH_CSV = ""
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET ÉS TOKENIZER --- (A korábbi BPE logikával)
class BPEDataset(Dataset):
    def __init__(self, recipes, tokenizer):
        self.recipes = recipes
        self.tok = tokenizer
    def __len__(self): return len(self.recipes)
    def __getitem__(self, i):
        s = [3] + self.tok.encode(self.recipes[i]["src"]).ids + [2] # 3:sos, 2:eos
        t = [3] + self.tok.encode(self.recipes[i]["tgt"]).ids + [2]
        return torch.tensor(s[:MAX_LEN]), torch.tensor(t[:MAX_LEN])

def collate_fn(batch):
    srcs = nn.utils.rnn.pad_sequence([b[0] for b in batch], padding_value=0)
    tgts = nn.utils.rnn.pad_sequence([b[1] for b in batch], padding_value=0)
    return srcs.to(DEVICE), tgts.to(DEVICE)

# --- 3. MODELL ARCHITEKTÚRA (Szigorúbb Encoder) ---
class RecipeTransformerV4(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, D_MODEL)
        self.pos = nn.Parameter(torch.randn(MAX_LEN, 1, D_MODEL))

        # 8 encoder réteg a precíz megértéshez, 4 decoder a folyékony szöveghez
        self.tf = nn.Transformer(
            d_model=D_MODEL, nhead=8,
            num_encoder_layers=8, num_decoder_layers=4,
            dropout=0.1
        )
        self.fc = nn.Linear(D_MODEL, vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.tf.generate_square_subsequent_mask(tgt.size(0)).to(DEVICE)
        s_emb = self.embed(src) + self.pos[:src.size(0)]
        t_emb = self.embed(tgt) + self.pos[:tgt.size(0)]
        return self.fc(self.tf(s_emb, t_emb, tgt_mask=tgt_mask))

# --- 4. SPECIÁLIS GENERÁLÓ FÜGGVÉNY A WEBAPPHOZ ---
def generate_v4_recipe(model, tokenizer, ingredients_list):
    model.eval()
    src_text = "ingredients: " + ", ".join(ingredients_list).lower()
    src_ids = [3] + tokenizer.encode(src_text).ids + [2]
    src = torch.tensor(src_ids).unsqueeze(1).to(DEVICE)
    res = [3]

    for _ in range(MAX_LEN):
        tgt = torch.tensor(res).unsqueeze(1).to(DEVICE)
        with torch.no_grad():
            output = model(src, tgt)
            next_id = output[-1, 0, :].argmax(-1).item()
            res.append(next_id)
            if next_id == 2: break

    full_text = tokenizer.decode(res)

    recipe_data = {"title": "N/A", "time": "N/A", "ingredients": [], "steps": "N/A"}
    try:
        # Precíz szétszedés a kulcsszavak alapján
        recipe_data["title"] = full_text.split("title:")[1].split("time:")[0].strip()
        recipe_data["time"] = full_text.split("time:")[1].split("ingredients:")[0].strip()
        recipe_data["ingredients"] = full_text.split("ingredients:")[1].split("steps:")[0].strip()
        recipe_data["steps"] = full_text.split("steps:")[1].strip()
    except:
        recipe_data["steps"] = full_text

    return recipe_data

def main_v4_smart():
    # 0. Mappa ellenőrzése
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # 1. Adatok betöltése
    all_recipes, all_weights = load_v4_data(DATA_PATH_CSV)

    #random.seed(42)
    indices = list(range(len(all_recipes)))
    random.shuffle(indices)
    split = int(len(all_recipes) * 0.9)
    train_recipes = [all_recipes[i] for i in indices[:split]]
    train_weights = [all_weights[i] for i in indices[:split]]

    # 2. Tokenizer előkészítése
    # Ha van már mentett tokenizer, használhatnánk, de a tisztaság kedvéért újratanítjuk
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train_from_iterator([r["src"] + " " + r["tgt"] for r in train_recipes],
                                  vocab_size=12000, special_tokens=["<pad>", "<unk>", "<eos>", "<sos>"])

    # Külön is elmentjük a tokentizert a Hugging Face Space-hez
    tokenizer_save_path = os.path.join(CHECKPOINT_DIR, "recipe_tokenizer.json")
    tokenizer.save(tokenizer_save_path)
    print(f"Tokenizer elmentve: {tokenizer_save_path}")

    # 3. Modell és Optimizer inicializálása
    model = RecipeTransformerV4(tokenizer.get_vocab_size()).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

    # --- RESUME LOGIKA ---
    start_epoch = 0
    history = {"epoch": [], "loss": [], "time": []}

    # Megkeressük a legutolsó mentést a mappában
    checkpoints = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')]
    if checkpoints:
        # Sorba rendezzük és kivesszük a legnagyobbat (pl. v4_structured_epoch_10.pth)
        latest = sorted(checkpoints, key=lambda x: int(x.split('_')[-1].split('.')[0]))[-1]
        checkpoint_path = os.path.join(CHECKPOINT_DIR, latest)
        print(f"Létező mentés betöltése: {checkpoint_path}")

        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch']
        history = checkpoint.get('history', history)
        # Ha a tokenizer is benne van, töltsük be azt is a biztonság kedvéért
        # (Bár most már külön is mentjük, de a kompatibilitás miatt jó ha itt is van)
        if 'tokenizer' in checkpoint:
            tokenizer = checkpoint['tokenizer']

    sampler = WeightedRandomSampler(train_weights, num_samples=len(train_weights), replacement=True)
    train_loader = DataLoader(BPEDataset(train_recipes, tokenizer),
                              batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_fn)

    # 4. Tanítási ciklus
    for epoch in range(start_epoch, 20):
        start_time = time.time()
        model.train()
        epoch_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for src, tgt in pbar:
            optimizer.zero_grad()
            output = model(src, tgt[:-1])
            loss = loss_fn(output.view(-1, output.size(-1)), tgt[1:].view(-1))
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = epoch_loss / len(train_loader)
        epoch_time = time.time() - start_time

        history["epoch"].append(epoch + 1)
        history["loss"].append(avg_loss)
        history["time"].append(epoch_time)

        # MENTÉS
        save_path = os.path.join(CHECKPOINT_DIR, f"v4_structured_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'tokenizer': tokenizer, # tokenizer object is also saved here for convenience
            'loss': avg_loss,
            'history': history
        }, save_path)

        print(f"\n[OK] Epoch {epoch+1} mentve. Loss: {avg_loss:.4f}")

        # PRÓBA GENERÁLÁS (Hiba esetén sem áll le)
        try:
            model.eval()
            res = generate_v4_recipe(model, tokenizer, ["flour", "egg", "milk"])
            print(f"Minta: {res['title']} | Idő: {res['time']} | Hozzávalók: {res['ingredients']} | Elkészítés: {res['steps']}")
        except Exception as e:
            print(f"Minta hiba: {e}")

if __name__ == "__main__":
    main_v4_smart() # Run this cell to (re)train and save the tokenizer

In [ ]:
import torch
import os
import torch.nn as nn
CHECKPOINT_DIR = ""
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

D_MODEL = 512
MAX_LEN = 400 
BATCH_SIZE = 32

class RecipeTransformerV4(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, D_MODEL)
        self.pos = nn.Parameter(torch.randn(MAX_LEN, 1, D_MODEL))

        # 8 encoder réteg a precíz megértéshez, 4 decoder a folyékony szöveghez
        self.tf = nn.Transformer(
            d_model=D_MODEL, nhead=8,
            num_encoder_layers=8, num_decoder_layers=4,
            dropout=0.1
        )
        self.fc = nn.Linear(D_MODEL, vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.tf.generate_square_subsequent_mask(tgt.size(0)).to(DEVICE)
        s_emb = self.embed(src) + self.pos[:src.size(0)]
        t_emb = self.embed(tgt) + self.pos[:tgt.size(0)]
        return self.fc(self.tf(s_emb, t_emb, tgt_mask=tgt_mask))



# 1. Elérési út beállítása
epoch_to_test = 20
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"v4_structured_epoch_{epoch_to_test}.pth")

# 2. Betöltés 
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

# 3. Tokenizer és Modell inicializálása a mentett adatokból
tokenizer = checkpoint['tokenizer']
vocab_size = tokenizer.get_vocab_size()

# Fontos: Pontosan ugyanazt az osztályt használd (RecipeTransformerV4), amivel tanítottál!
model = RecipeTransformerV4(vocab_size).to(DEVICE)
model.load_state_dict(checkpoint['model'])
model.eval() # Kikapcsoljuk a Dropout-ot és a Gradiens számítást

print(f"Sikeresen betöltve: {checkpoint_path}")
print(f"Modell Loss értéke a 20. epochnál: {checkpoint['loss']:.4f}")

In [ ]:
def generate_v4_recipe(model, tokenizer, ingredients_list):
    model.eval()
    src_text = "ingredients: " + ", ".join(ingredients_list).lower()
    src_ids = [3] + tokenizer.encode(src_text).ids + [2]
    src = torch.tensor(src_ids).unsqueeze(1).to(DEVICE)
    res = [3]

    for _ in range(MAX_LEN):
        tgt = torch.tensor(res).unsqueeze(1).to(DEVICE)
        with torch.no_grad():
            output = model(src, tgt)
            next_id = output[-1, 0, :].argmax(-1).item()
            res.append(next_id)
            if next_id == 2: break

    full_text = tokenizer.decode(res)

    recipe_data = {"title": "N/A", "time": "N/A", "ingredients": [], "steps": "N/A"}
    try:
        # Precíz szétszedés a kulcsszavak alapján
        recipe_data["title"] = full_text.split("title:")[1].split("time:")[0].strip()
        recipe_data["time"] = full_text.split("time:")[1].split("ingredients:")[0].strip()
        recipe_data["ingredients"] = full_text.split("ingredients:")[1].split("steps:")[0].strip()
        recipe_data["steps"] = full_text.split("steps:")[1].strip()
    except:
        recipe_data["steps"] = full_text

    return recipe_data


def test_my_model(ingredients):
    # A korábban megírt v4-es generáló függvényt használjuk
    recipe = generate_v4_recipe(model, tokenizer, ingredients)
    ing_set = {i.strip() for i in recipe['ingredients'].split(',')}
    print(recipe['steps'])
    print("tenyleges hozzavalok: ", recipe['ingredients'])
    print("-" * 30)
    print(f"🍳 RECEPT JAVASLAT")
    print("-" * 30)
    print(f"NAME: {recipe['title'].upper()}")
    print(f"TIME: {recipe['time']}")
    print(f"INGREDIENTTS: {','.join(ing_set)}")
    print("-" * 30)
    print(f"STEPS:\n{recipe['steps']}")
    print("-" * 30)

# Próbáld ki valami extrával!
test_my_model(['chicken', 'rice', 'garlic'])

In [ ]:


def generate_v4_with_penalty(model, tokenizer, ingredients_list, penalty=1.5, temperature=0.2):
    model.eval()
    src_text = "ingredients: " + ", ".join(ingredients_list).lower()
    src_ids = [3] + tokenizer.encode(src_text).ids + [2]
    src = torch.tensor(src_ids).unsqueeze(1).to(DEVICE)
    res = [3] # SOS token

    for _ in range(MAX_LEN):
        tgt = torch.tensor(res).unsqueeze(1).to(DEVICE)
        with torch.no_grad():
            output = model(src, tgt)
            # Csak az utolsó időköz logits-eire van szükségünk
            logits = output[-1, 0, :]

            # --- REPETITION PENALTY LOGIKA ---
            for token_id in set(res):
                # Ha a logit pozitív, osztjuk a büntetéssel (csökken a valószínűség)
                # Ha negatív, szorozzuk (még kisebb lesz)
                if logits[token_id] > 0:
                    logits[token_id] /= penalty
                else:
                    logits[token_id] *= penalty

            # A büntetett logits-ek alapján választjuk a legvalószínűbbet
            probs = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            res.append(next_id)

            if next_id == 2: break # EOS

    full_text = tokenizer.decode(res)

    # Strukturált szétszedés (marad a korábbi logika)
    recipe_data = {"title": "N/A", "time": "N/A", "ingredients": [], "steps": "N/A"}
    try:
        recipe_data["title"] = full_text.split("title:")[1].split("time:")[0].strip()
        recipe_data["time"] = full_text.split("time:")[1].split("ingredients:")[0].strip()
        recipe_data["ingredients"] = full_text.split("ingredients:")[1].split("steps:")[0].strip()
        recipe_data["steps"] = full_text.split("steps:")[1].strip()
    except:
        recipe_data["steps"] = full_text

    return recipe_data

for i in range (20):
  recipe = generate_v4_with_penalty(model, tokenizer, ['eggs', 'flour', 'milk', 'sugar'])
  print(recipe)

In [ ]:
import torch
from tokenizers import ByteLevelBPETokenizer, Tokenizer
import os
import gradio as gr
import json
from huggingface_hub import hf_hub_download

# --- CONFIGURATION (Must match training!) ---
D_MODEL = 512
MAX_LEN = 400

# Hugging Face Model Details
HF_USERNAME = "l-e-m-i"
REPO_NAME = "szakdoga-scratch-v2"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

# The model architecture class must be defined here!
class RecipeTransformerV4(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, D_MODEL)
        self.pos = torch.nn.Parameter(torch.randn(MAX_LEN, 1, D_MODEL))
        self.tf = torch.nn.Transformer(
            d_model=D_MODEL, nhead=8,
            num_encoder_layers=8, num_decoder_layers=4,
            dropout=0.1
        )
        self.fc = torch.nn.Linear(D_MODEL, vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.tf.generate_square_subsequent_mask(tgt.size(0)).to(src.device)
        s_emb = self.embed(src) + self.pos[:src.size(0)]
        t_emb = self.embed(tgt) + self.pos[:tgt.size(0)]
        return self.fc(self.tf(s_emb, t_emb, tgt_mask=tgt_mask))

# --- Load Model and Tokenizer ---
# This function will be used by the Hugging Face Space.
def load_model_for_app():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Downloading model from {REPO_ID}...")

    # Only download the .pth file, since it contains the tokenizer too!
    model_path_local = hf_hub_download(repo_id=REPO_ID, filename="pytorch_model.pth")

    # Load the full checkpoint
    checkpoint = torch.load(model_path_local, map_location=device, weights_only=False)

    # Load Tokenizer directly from the checkpoint
    tokenizer = checkpoint['tokenizer']
    vocab_size = tokenizer.get_vocab_size()

    # Initialize Model and load weights
    model = RecipeTransformerV4(vocab_size).to(device)

    # Check if the state_dict is nested under 'model' key
    state_dict = checkpoint.get('model', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()

    print(f"Model and tokenizer loaded successfully on {device}!")
    return model, tokenizer, device

# --- Recipe Generation Function (with repetition penalty) ---
def generate_v4_with_penalty(model, tokenizer, ingredients_str, device, penalty=1.5, temperature=0.2, max_len=MAX_LEN):
    model.eval()
    src_text = "ingredients: " + ingredients_str.lower()
    src_ids = [3] + tokenizer.encode(src_text).ids + [2] # 3: SOS, 2: EOS
    src = torch.tensor(src_ids).unsqueeze(1).to(device)
    res = [3] # SOS token

    for _ in range(max_len):
        tgt = torch.tensor(res).unsqueeze(1).to(device)
        with torch.no_grad():
            output = model(src, tgt)
            logits = output[-1, 0, :]

            for token_id in set(res):
                if logits[token_id] > 0:
                    logits[token_id] /= penalty
                else:
                    logits[token_id] *= penalty

            probs = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            res.append(next_id)

            if next_id == 2: break # EOS

    full_text = tokenizer.decode(res)

    recipe_data = {"title": "N/A", "time": "N/A", "ingredients": "N/A", "steps": "N/A", "raw_text": full_text}

    try:
        title_start = full_text.find("title:")
        time_start = full_text.find("time:")
        ing_start = full_text.find("ingredients:")
        steps_start = full_text.find("steps:")

        if title_start != -1:
            title_end = min(idx for idx in [time_start, ing_start, steps_start, len(full_text)] if idx != -1 and idx > title_start)
            recipe_data["title"] = full_text[title_start + len("title:"):title_end].strip()

        if time_start != -1:
            time_end = min(idx for idx in [ing_start, steps_start, len(full_text)] if idx != -1 and idx > time_start)
            recipe_data["time"] = full_text[time_start + len("time:"):time_end].strip()

        if ing_start != -1:
            ing_end = min(idx for idx in [steps_start, len(full_text)] if idx != -1 and idx > ing_start)
            recipe_data["ingredients"] = full_text[ing_start + len("ingredients:"):ing_end].strip()

        if steps_start != -1:
            recipe_data["steps"] = full_text[steps_start + len("steps:"):].strip()

    except Exception as e:
        print(f"Error during parsing: {e}")
        recipe_data["steps"] = full_text # Fallback
        recipe_data["ingredients"] = "N/A"

    return recipe_data


# --- Gradio Interface ---

# Load model on startup
model, tokenizer, device = load_model_for_app()

def generate_recipe_for_gradio(ingredients_input):
    # Input can be a comma-separated string, or a list; here we expect a list
    ingredients_list = [i.strip() for i in ingredients_input.split(',') if i.strip()]

    if not ingredients_list:
        return json.dumps({"error": "Please provide ingredients!"}, indent=2)

    # Default values for penalty and temperature
    repetition_penalty_val = 1.5
    temperature_val = 0.2

    # Generate using the fine-tuned function
    generated_data = generate_v4_with_penalty(
        model, tokenizer, ", ".join(ingredients_list),
        device,
        penalty=repetition_penalty_val,
        temperature=temperature_val
    )

    # Prepare dictionary for JSON output
    recipe_json_output = {
        "title": generated_data["title"].upper(),
        "time": generated_data["time"],
        "ingredients": generated_data["ingredients"],
        "steps": generated_data["steps"],
        "raw_text": generated_data["raw_text"]
    }

    return json.dumps(recipe_json_output, indent=2)


iface = gr.Interface(
    fn=generate_recipe_for_gradio,
    inputs=[
        gr.Textbox(label="Ingredients (comma-separated)", placeholder="e.g. chicken, rice, garlic")
    ],
    outputs=[
        gr.JSON(label="Generated Recipe (JSON)")
    ],
    title="AI Recipe Generator",
    description="Enter ingredients and generate a recipe!",
    examples=[["chicken, broccoli, pasta"], ["eggs, flour, milk, sugar"]],
)

iface.launch()


In [ ]:
!pip install transformers datasets accelerate sentencepiece evaluate rouge_score bert_score -q
# ==============================================================================
# KIÉRTÉKELŐ SCRIPT A SAJÁT TRANSFORMER (v4_structured) MODELLHEZ
# ==============================================================================

import os
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import numpy as np
import pandas as pd
import re
import nltk
from tqdm import tqdm
from datasets import Dataset

nltk.download("punkt", quiet=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. KONFIGURÁCIÓ ÉS ARCHITEKTÚRA ---
CHECKPOINT_DIR = ""
DATA_PATH_CSV = ""
D_MODEL = 512
MAX_LEN = 400

class RecipeTransformerV4(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, D_MODEL)
        self.pos = nn.Parameter(torch.randn(MAX_LEN, 1, D_MODEL))
        self.tf = nn.Transformer(
            d_model=D_MODEL, nhead=8,
            num_encoder_layers=8, num_decoder_layers=4,
            dropout=0.1
        )
        self.fc = nn.Linear(D_MODEL, vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.tf.generate_square_subsequent_mask(tgt.size(0)).to(DEVICE)
        s_emb = self.embed(src) + self.pos[:src.size(0)]
        t_emb = self.embed(tgt) + self.pos[:tgt.size(0)]
        return self.fc(self.tf(s_emb, t_emb, tgt_mask=tgt_mask))

# --- 2. MODELL ÉS TOKENIZER BETÖLTÉSE (A Te kódod alapján) ---
print("Modell és Tokenizer betöltése...")
epoch_to_test = 20
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"v4_structured_epoch_{epoch_to_test}.pth")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

tokenizer = checkpoint['tokenizer']
vocab_size = tokenizer.get_vocab_size()

model = RecipeTransformerV4(vocab_size).to(DEVICE)
model.load_state_dict(checkpoint['model'])
model.eval() # Gradiens és Dropout kikapcsolása
print(f"Sikeresen betöltve: {checkpoint_path} (Loss: {checkpoint.get('loss', 0):.4f})")

# --- 3. TESZT ADATOK ELŐKÉSZÍTÉSE ---
print("\nTeszt adatok betöltése...")
df = pd.read_csv(DATA_PATH_CSV)
df["ingredients"] = df["ingredients"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["steps"] = df["steps"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df = df[df['steps'].map(len) >= 4]
df = df[df['ingredients'].map(len) >= 4]

# Ugyanaz a random_state (42), mint a finetuned modellnél, hogy a teszthalmaz azonos legyen!
if len(df) > 30000:
    df = df.sample(30000, random_state=42)

dataset = Dataset.from_pandas(df)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset_split["test"]

# --- 4. GENERÁLÓ FÜGGVÉNY (SMART DECODER) ---
def generate_v4_smart(model, tokenizer, ingredients_list, temperature=0.7, top_k=20):
    src_text = "ingredients: " + ", ".join(ingredients_list).lower()

    # Kódolás a BPE tokenizerrel (3 = SOS, 2 = EOS tokenek a Te korábbi logikád alapján)
    src_ids = [3] + tokenizer.encode(src_text).ids + [2]
    src = torch.tensor(src_ids).unsqueeze(1).to(DEVICE)
    res = [3]

    for _ in range(MAX_LEN):
        tgt = torch.tensor(res).unsqueeze(1).to(DEVICE)
        with torch.no_grad():
            output = model(src, tgt)
            # Logitok kinyerése és hőmérséklet (Temperature) alkalmazása
            logits = output[-1, 0, :] / temperature
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            probs = F.softmax(top_k_logits, dim=-1)

            # Mintavételezés
            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_id = top_k_indices[next_idx].item()
            res.append(next_id)

            if next_id == 2: # EOS token elérése
                break

    return tokenizer.decode(res, skip_special_tokens=True)

# --- 5. JAVÍTOTT ROBUST PARSER A HOZZÁVALÓKHOZ (SET ALAPÚ DEDUPLIKÁCIÓVAL) ---
def extract_ingredients_robust(text):
    text = text.lower().strip()
    match = re.search(r"ingredients[:\s](.*?)steps[:\s]", text, re.DOTALL)
    if not match:
        match = re.search(r"ingredients[:\s](.*)$", text, re.DOTALL)
    if match:
        raw = match.group(1).strip()
        clean = raw.replace('\n', ',').replace('-', ',').replace('•', ',')

        # Nyers lista a vesszők mentén
        raw_list = [i.strip() for i in clean.split(',') if len(i.strip()) > 1]

        # SET (Halmaz) konverzió a duplikációk eltávolítására, majd vissza lista
        unique_ingredients = list(set(raw_list))
        return unique_ingredients

    return []

# --- 6. FŐ KIÉRTÉKELŐ CIKLUS ---
def evaluate_scratch_v4(num_samples=100):
    test_subset = test_dataset.select(range(min(len(test_dataset), num_samples)))

    predictions = []
    references = []
    inputs_raw = []

    print(f"\nGenerálás {len(test_subset)} teszt mintán...")

    for row in tqdm(test_subset):
        real_ingredients = row["ingredients"]

        # Referencia szöveg összeállítása a metrikákhoz
        target_text = (
            f"title: {str(row['name']).lower()} "
            f"time: {str(row['minutes'])} mins "
            f"ingredients: {', '.join(real_ingredients).lower()} "
            f"steps: {'; '.join(row['steps']).lower()}"
        )

        inputs_raw.append(", ".join(real_ingredients))
        references.append(target_text)

        # Generálás a betöltött modellel
        pred_text = generate_v4_smart(model, tokenizer, real_ingredients)
        predictions.append(pred_text)

    print("\nMetrikák számítása (ROUGE, BERTScore)...")
    bertscore = evaluate.load("bertscore")
    rouge = evaluate.load("rouge")

    rouge_results = rouge.compute(predictions=predictions, references=references)
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en", device=DEVICE)
    avg_bert = np.mean(bert_results['f1'])

    precision_scores, recall_scores = [], []
    valid_parsing_count = 0

    for i, pred in enumerate(predictions):
        gen_ings = extract_ingredients_robust(pred)
        target_ings = extract_ingredients_robust(references[i])

        if len(gen_ings) == 0:
            precision_scores.append(0.0)
            recall_scores.append(0.0)
            continue

        valid_parsing_count += 1
        hits = sum(1 for gen_ing in gen_ings for target_ing in target_ings if gen_ing in target_ing or target_ing in gen_ing)
        #precision_scores.append(hits / len(gen_ings) if len(gen_ings) > 0 else 0)
        #recall_scores.append(hits / len(target_ings) if len(target_ings) > 0 else 0)

        # --- BOMBABIZTOS PRECISION / RECALL SZÁMÍTÁS ---
        # Precizitás: A kigenerált hozzávalók közül hánynak van legalább 1 párja a referenciában?
        matched_gen = sum(1 for g in gen_ings if any(g in t or t in g for t in target_ings))
        precision_scores.append(matched_gen / len(gen_ings) if len(gen_ings) > 0 else 0)

        # Visszahívás: A referencia hozzávalók közül hányat "fedett le" a gép?
        matched_tgt = sum(1 for t in target_ings if any(g in t or t in g for g in gen_ings))
        recall_scores.append(matched_tgt / len(target_ings) if len(target_ings) > 0 else 0)


    avg_precision = np.mean(precision_scores) if precision_scores else 0.0
    avg_recall = np.mean(recall_scores) if recall_scores else 0.0
    r1 = rouge_results['rouge1'] if isinstance(rouge_results['rouge1'], float) else rouge_results['rouge1'].mid.fmeasure

    print("\n" + "="*40)
    print("EREDMÉNYEK (SCRATCH V4 - EPOCH 20)")
    print("="*40)
    print(f"Sikeres formátum: {valid_parsing_count} / {len(predictions)}")
    print("-" * 40)
    print(f"ROUGE-1:      {r1*100:.2f}%")
    print(f"BERTScore:    {avg_bert*100:.2f}%")
    print("-" * 40)
    print(f"Precision:    {avg_precision*100:.2f}%")
    print(f"Recall:       {avg_recall*100:.2f}%")
    print("="*40)

    # Eredmények mentése
    out_file = os.path.join(CHECKPOINT_DIR, "scratch_epoch20_results.csv")
    pd.DataFrame({"Input": inputs_raw, "Target": references, "Generated": predictions}).to_csv(out_file, index=False)
    print(f"Részletes predikciók mentve: {out_file}")

# Futtatás (Kezdésnek érdemes 100 mintán futtatni, hogy ne tartson órákig)
evaluate_scratch_v4(num_samples=100)